# 🏆 Web Scraping — Notebook 5
## Final Course Project: Multi-Source Enriched Data Pipeline

---

## 🎯 Project Goal

Build a **fully enriched, multi-table scraper** that combines every skill from all 4 notebooks:

```
PHASE 1 — Scrape Quotes  (requests + BS4, paginated, Session)
   └── quotes.toscrape.com → all quotes + tags → quotes table

PHASE 2 — Enrich with Author Bios  (requests + BS4, detail pages)
   └── For each unique author → visit their page → scrape bio, born date
   └── authors table (linked to quotes via author name)

PHASE 3 — JS Fallback Demo  (Selenium)
   └── Same site's /js/ version → smart_fetch auto-detects and uses Selenium

PHASE 4 — Multi-Table SQL + Export
   └── JOIN quotes + authors → enriched_quotes.csv
   └── Final analysis across both tables
```

### Skills Used in This Project

```
From NB1:  requests, BS4, find_all, CSS selectors, pagination
From NB2:  Selenium, WebDriverWait, smart_fetch fallback
From NB3:  Session, retry, logging, SQLite, dedup, checkpoint, validate, robots.txt
From NB4:  parse → validate → save loop, context manager DB, tqdm, quality report
NEW:       Multi-table DB, detail page enrichment, SQL JOIN
```

### Architecture

```
                quotes.toscrape.com
                        │
         ┌──────────────┴──────────────┐
         │                             │
    /page/1..10               /author/<name>/
   (listing pages)           (detail pages)
         │                             │
   quotes table               authors table
   ─────────────               ─────────────
   id, quote, author, tags    id, name, born_date, born_loc, bio
         │                             │
         └──────────── JOIN ───────────┘
                         │
               enriched_quotes.csv
```

## ⚙️ Setup — All Imports & Config

All configuration in one place — the production way.

In [ ]:
# SETUP — All imports and configuration

import requests
import re
import csv
import json
import os
import time
import random
import logging
import sqlite3
import hashlib
from datetime import datetime
from urllib.parse import urljoin, urlparse
from urllib.robotparser import RobotFileParser
from bs4 import BeautifulSoup

# Selenium
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from webdriver_manager.chrome import ChromeDriverManager

try:
    from tqdm import tqdm
    TQDM = True
except ImportError:
    TQDM = False

# ── Config ──
BASE_URL        = 'https://quotes.toscrape.com'
START_URL       = 'https://quotes.toscrape.com/page/1/'
JS_URL          = 'https://quotes.toscrape.com/js/'
DB_FILE         = 'final_project.db'
CSV_FILE        = 'enriched_quotes.csv'
CHECKPOINT_FILE = 'final_checkpoint.json'
DELAY_MIN, DELAY_MAX = 0.8, 1.5
MAX_RETRIES     = 3

HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0'}

# ── Logging ──
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-8s | %(message)s',
    datefmt='%H:%M:%S',
    handlers=[
        logging.FileHandler('final_project.log', mode='w'),
        logging.StreamHandler()
    ]
)
log = logging.getLogger('final_project')

# ── robots.txt ──
def can_scrape(target_url):
    parsed = urlparse(target_url)
    rp = RobotFileParser()
    rp.set_url(f'{parsed.scheme}://{parsed.netloc}/robots.txt')
    try:
        rp.read()
        return rp.can_fetch('*', target_url)
    except Exception:
        return True

allowed = can_scrape(START_URL)
log.info(f'robots.txt: {"ALLOWED" if allowed else "BLOCKED"}')
print(f'robots.txt check : {"✅ ALLOWED" if allowed else "❌ BLOCKED"}')
print('Setup complete!')


## 🗄️ Multi-Table Database Design

This project uses **two linked tables** — a real relational database pattern:

```sql
quotes table               authors table
────────────────────────   ─────────────────────────────────
id       INTEGER PK        id         INTEGER PK
quote    TEXT UNIQUE       name       TEXT UNIQUE
author   TEXT          ←── born_date  TEXT
tags     TEXT (JSON)       born_loc   TEXT
page_num INTEGER           bio        TEXT
scraped_at TEXT            scraped_at TEXT
```

### Linking Tables
`quotes.author` references `authors.name`.
This lets us JOIN to get: **quote + author bio in one row**.

```sql
SELECT q.quote, q.author, a.born_date, a.born_loc
FROM quotes q
JOIN authors a ON q.author = a.name;
```

In [ ]:
# DB SETUP — Two tables

def init_db():
    with sqlite3.connect(DB_FILE) as conn:
        conn.execute('''
            CREATE TABLE IF NOT EXISTS quotes (
                id         INTEGER PRIMARY KEY AUTOINCREMENT,
                quote      TEXT UNIQUE,
                author     TEXT,
                tags       TEXT,
                page_num   INTEGER,
                scraped_at TEXT
            )
        ''')
        conn.execute('''
            CREATE TABLE IF NOT EXISTS authors (
                id         INTEGER PRIMARY KEY AUTOINCREMENT,
                name       TEXT UNIQUE,
                born_date  TEXT,
                born_loc   TEXT,
                bio        TEXT,
                scraped_at TEXT
            )
        ''')
    log.info(f'Database ready: {DB_FILE} (2 tables: quotes + authors)')
    print(f'DB ready: {DB_FILE}')
    print('Tables: quotes, authors')


# ── Session factory ──
def make_session():
    s = requests.Session()
    s.headers.update(HEADERS)
    return s


# ── Fetch with retry (uses Session) ──
def fetch(session, url):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            r = session.get(url, timeout=10)
            r.raise_for_status()
            return BeautifulSoup(r.text, 'html.parser')
        except requests.exceptions.HTTPError as e:
            log.error(f'HTTP {e.response.status_code} [{attempt}]: {url}')
            if e.response.status_code == 404: return None
        except requests.exceptions.RequestException as e:
            log.warning(f'Attempt {attempt} failed: {e}')
            if attempt < MAX_RETRIES: time.sleep(2 ** (attempt - 1))
    return None


# ── Safe helpers ──
def safe_text(tag, default='N/A'):
    return tag.get_text(strip=True) if tag else default


init_db()


## 🧠 Smart Fetcher — requests + Selenium Fallback

The smart fetcher from NB3, now fully integrated into the project.

```
smart_fetch(url, js_indicator)
         │
         ├── requests.get() → check for js_indicator
         │         │
         │    Found? ──YES──> return soup  (fast path, no browser)
         │         │
         │    Not found? ─> Selenium launches ─> page_source ─> return soup
         │
         └── Exception? ─> retry → log → return None
```

We'll use this for **Phase 3** where we scrape the JS-rendered version of the site.

In [ ]:
# SMART FETCHER (requests → Selenium fallback)

def get_driver(headless=True):
    opts = Options()
    if headless: opts.add_argument('--headless')
    opts.add_argument('--no-sandbox')
    opts.add_argument('--disable-dev-shm-usage')
    opts.add_argument('--disable-blink-features=AutomationControlled')
    opts.add_argument('--window-size=1920,1080')
    opts.add_argument(f'user-agent={HEADERS["User-Agent"]}')
    opts.add_experimental_option('excludeSwitches', ['enable-logging'])
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=opts)


def smart_fetch(session, url, js_indicator=None, timeout=10):
    """
    Try requests first; fall back to Selenium if JS indicator missing.
    Returns BeautifulSoup or None.
    """
    # Step 1: Try requests
    try:
        r = session.get(url, timeout=timeout)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, 'html.parser')
        if js_indicator is None or soup.select_one(js_indicator):
            log.info(f'[requests] {url}')
            return soup, 'requests'
        log.info(f'[requests] JS indicator missing → Selenium fallback')
    except requests.exceptions.RequestException as e:
        log.warning(f'[requests] Failed: {e} → Selenium fallback')

    # Step 2: Selenium fallback
    driver = get_driver(headless=True)
    try:
        driver.get(url)
        if js_indicator:
            WebDriverWait(driver, timeout).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, js_indicator))
            )
        else:
            time.sleep(3)
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        log.info(f'[selenium] {url}')
        return soup, 'selenium'
    except Exception as e:
        log.error(f'[selenium] Failed: {e}')
        return None, None
    finally:
        driver.quit()


# ── Test: static page (requests should work) ──
session = make_session()
soup, method = smart_fetch(session, START_URL, js_indicator='div.quote')
print(f'Static page fetched via: [{method}]')
print(f'Quotes found: {len(soup.find_all("div", class_="quote"))}')


## 📖 Phase 1 — Scrape All Quotes (All Pages)

Scrape every quote + author + tags from all 10 pages.
We also collect the **author page URL** from each quote card — needed for Phase 2.

```html
<!-- Each quote card has an author page link -->
<small class="author">Albert Einstein
  <a href="/author/albert-einstein">(about)</a>
</small>
```

We'll store the author href so we can visit each author's bio page in Phase 2.

In [ ]:
# PHASE 1 — Scrape All Quotes

def parse_quotes_page(soup, page_num):
    """Extract quotes + author page URLs from one listing page."""
    now    = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    quotes = []
    author_urls = {}   # {author_name: author_page_url}

    for box in soup.find_all('div', class_='quote'):
        text_tag   = box.find('span', class_='text')
        author_tag = box.find('small', class_='author')
        tag_links  = box.find_all('a', class_='tag')
        about_link = box.find('a', href=lambda h: h and '/author/' in h)

        quote  = safe_text(text_tag)
        author = safe_text(author_tag)
        tags   = json.dumps([safe_text(t) for t in tag_links])
        a_url  = urljoin(BASE_URL, about_link['href']) if about_link else None

        # Validate
        if quote == 'N/A' or author == 'N/A':
            log.warning(f'Invalid quote on page {page_num}: missing quote or author')
            continue

        quotes.append({'quote': quote, 'author': author,
                       'tags': tags, 'page_num': page_num, 'scraped_at': now})
        if author not in author_urls and a_url:
            author_urls[author] = a_url

    return quotes, author_urls


def save_quotes(quotes):
    saved = skipped = 0
    with sqlite3.connect(DB_FILE) as conn:
        for q in quotes:
            c = conn.execute(
                'INSERT OR IGNORE INTO quotes (quote,author,tags,page_num,scraped_at)'
                ' VALUES (?,?,?,?,?)',
                (q['quote'], q['author'], q['tags'], q['page_num'], q['scraped_at'])
            )
            if c.rowcount > 0: saved += 1
            else: skipped += 1
    return saved, skipped


# ── Run Phase 1 ──
all_author_urls = {}   # Collect author URLs across all pages
total_quotes    = 0
current_url     = START_URL
page_num        = 1

log.info('PHASE 1: Scraping quotes...')
print('Phase 1: Scraping quotes...')

pbar = tqdm(total=10, desc='Pages', unit='pg') if TQDM else None

while current_url:
    soup = fetch(session, current_url)
    if not soup: break

    quotes, author_urls = parse_quotes_page(soup, page_num)
    saved, skipped      = save_quotes(quotes)
    total_quotes       += saved
    all_author_urls.update(author_urls)   # Collect author URLs

    if pbar: pbar.set_postfix(saved=total_quotes); pbar.update(1)
    else: log.info(f'Page {page_num:2d}: {saved} saved | {len(all_author_urls)} unique authors')

    # Next page
    next_li     = soup.find('li', class_='next')
    current_url = urljoin(current_url, next_li.find('a')['href']) if next_li else None
    page_num   += 1
    if current_url: time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

if pbar: pbar.close()

print()
print(f'Phase 1 complete!')
print(f'  Quotes saved    : {total_quotes}')
print(f'  Unique authors  : {len(all_author_urls)}')
print('Sample author URLs:')
for name, url in list(all_author_urls.items())[:3]:
    print(f'  {name:25s} -> {url}')


## 👤 Phase 2 — Detail Page Enrichment: Author Bios

This is a very common real-world pattern: **scrape a list page, then visit each detail page.**

```
LISTING SCRAPER           DETAIL SCRAPER
─────────────────         ─────────────────────────────────
quotes.toscrape.com/  →   quotes.toscrape.com/author/albert-einstein/
                                │
                          Scrape:
                          - born_date ("June 14, 1879")
                          - born_loc  ("Ulm, Germany")
                          - description (bio paragraph)
```

### Author Page HTML
```html
<div class="author-details">
  <h3 class="author-title">Albert Einstein</h3>
  <span class="author-born-date">June 14, 1879</span>
  <span class="author-born-location">in Ulm, Germany</span>
  <div class="author-description">...</div>
</div>
```

In [ ]:
# PHASE 2 — Scrape Author Bio Pages

def parse_author_page(soup, author_name):
    """Extract bio info from an author detail page."""
    born_date = safe_text(soup.find('span', class_='author-born-date'))
    born_loc  = safe_text(soup.find('span', class_='author-born-location'))
    bio_div   = soup.find('div', class_='author-description')
    bio       = bio_div.get_text(strip=True)[:500] if bio_div else 'N/A'  # First 500 chars

    return {
        'name'      : author_name,
        'born_date' : born_date,
        'born_loc'  : born_loc.lstrip('in ').strip(),
        'bio'       : bio,
        'scraped_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    }


def save_author(author):
    with sqlite3.connect(DB_FILE) as conn:
        c = conn.execute(
            'INSERT OR IGNORE INTO authors (name,born_date,born_loc,bio,scraped_at)'
            ' VALUES (?,?,?,?,?)',
            (author['name'], author['born_date'], author['born_loc'],
             author['bio'], author['scraped_at'])
        )
        return c.rowcount > 0


# ── Run Phase 2 ──
log.info(f'PHASE 2: Scraping {len(all_author_urls)} author pages...')
print(f'Phase 2: Enriching {len(all_author_urls)} authors with bio data...')
print()

authors_saved = 0
author_list   = list(all_author_urls.items())

pbar2 = tqdm(author_list, desc='Authors', unit='author') if TQDM else author_list

for author_name, author_url in pbar2:
    soup = fetch(session, author_url)
    if not soup:
        log.warning(f'Could not fetch author page: {author_name}')
        continue

    author_data = parse_author_page(soup, author_name)
    saved       = save_author(author_data)
    if saved: authors_saved += 1

    if not TQDM:
        log.info(f'Author scraped: {author_name} | born: {author_data["born_date"]}')

    time.sleep(random.uniform(0.5, 1.2))   # Polite delay

if TQDM: pbar2.close()

print()
print(f'Phase 2 complete!')
print(f'  Authors saved : {authors_saved}')

# Preview a few authors
with sqlite3.connect(DB_FILE) as conn:
    conn.row_factory = sqlite3.Row
    rows = conn.execute('SELECT name, born_date, born_loc FROM authors LIMIT 5').fetchall()

print()
print('Sample authors in DB:')
print(f'{"Name":<25} {"Born Date":<20} {"Location"}')
print('-' * 65)
for r in rows:
    print(f'{r["name"]:<25} {r["born_date"]:<20} {r["born_loc"]}')


## 🤖 Phase 3 — Selenium Fallback Demo (JS-Rendered Page)

The same site has a JS version: `quotes.toscrape.com/js/`
When `requests` fetches it, the quotes aren't there — they're injected by JavaScript.
`smart_fetch()` automatically detects this and switches to Selenium.

```
smart_fetch('/js/', js_indicator='div.quote')
      │
      ├── requests → soup → look for div.quote → NOT FOUND
      │
      └── Selenium → JS runs → quotes appear → found! → return soup
```

> This demonstrates the full **requests → Selenium fallback** chain in a real scenario.

In [ ]:
# PHASE 3 — JS Page via smart_fetch (Selenium fallback)

print('Phase 3: Fetching JS-rendered page...')
print()

# Try requests first (will fail to find quotes)
r = session.get(JS_URL)
soup_requests = BeautifulSoup(r.text, 'html.parser')
quotes_via_requests = len(soup_requests.find_all('div', class_='quote'))
print(f'via requests only → quotes found: {quotes_via_requests}  (empty — JS not executed)')

# Now use smart_fetch (will detect 0 quotes → launch Selenium)
print()
print('Launching smart_fetch...')
soup_smart, method = smart_fetch(session, JS_URL, js_indicator='div.quote')

if soup_smart:
    quotes_via_smart = soup_smart.find_all('div', class_='quote')
    print(f'via smart_fetch → [{method}] → quotes found: {len(quotes_via_smart)}')
    print()

    # Parse and show first 3
    print('First 3 quotes from JS page:')
    for box in quotes_via_smart[:3]:
        text   = box.find('span', class_='text')
        author = box.find('small', class_='author')
        print(f'  {safe_text(author):20s}: {safe_text(text)[:55]}...')
else:
    print('smart_fetch failed completely')

print()
print('Phase 3 complete — Selenium fallback works!')


## 🔗 Phase 4 — SQL JOIN: Combine Quotes + Author Bios

Now that we have data in two tables, **JOIN** them to create an enriched dataset.

```sql
SELECT
    q.quote,
    q.author,
    q.tags,
    a.born_date,
    a.born_loc,
    a.bio
FROM quotes q
LEFT JOIN authors a ON q.author = a.name
ORDER BY q.author;
```

**LEFT JOIN** means: include ALL quotes, even if no author bio was found.
Regular JOIN would exclude quotes where the author page scrape failed.

### LEFT JOIN vs INNER JOIN
```
quotes    LEFT JOIN authors     All quotes + bio where available
quotes   INNER JOIN authors     Only quotes WITH bio (may lose some quotes)
```

In [ ]:
# PHASE 4 — SQL JOIN: enriched dataset

def get_enriched_quotes():
    """JOIN quotes + authors to produce enriched rows."""
    with sqlite3.connect(DB_FILE) as conn:
        conn.row_factory = sqlite3.Row
        rows = conn.execute('''
            SELECT
                q.id,
                q.quote,
                q.author,
                q.tags,
                q.page_num,
                a.born_date,
                a.born_loc,
                SUBSTR(a.bio, 1, 150) AS bio_snippet
            FROM quotes q
            LEFT JOIN authors a ON q.author = a.name
            ORDER BY q.author, q.id
        ''').fetchall()
    return [dict(r) for r in rows]


enriched = get_enriched_quotes()

print(f'Enriched rows: {len(enriched)}')
print()

# Check coverage
with_bio    = sum(1 for r in enriched if r['born_date'] and r['born_date'] != 'N/A')
without_bio = len(enriched) - with_bio
print(f'Quotes with author bio : {with_bio}')
print(f'Quotes without bio     : {without_bio}')
print(f'Bio coverage           : {with_bio/len(enriched)*100:.1f}%')
print()

# Show 3 enriched rows
print('Sample enriched rows:')
print('-' * 75)
for row in enriched[:3]:
    print(f'Quote  : {row["quote"][:60]}...')
    print(f'Author : {row["author"]}')
    print(f'Born   : {row["born_date"]} in {row["born_loc"]}')
    print(f'Tags   : {row["tags"]}')
    print(f'Bio    : {(row["bio_snippet"] or "")[:80]}...')
    print()


## 💾 Export & Data Quality Check

Export the enriched dataset to CSV and run a quality report before calling it done.

In [ ]:
# EXPORT TO CSV + DATA QUALITY REPORT

# ── Export ──
if enriched:
    with open(CSV_FILE, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=enriched[0].keys())
        writer.writeheader()
        writer.writerows(enriched)
    print(f'Exported {len(enriched)} rows → {CSV_FILE}')

# ── Data Quality Report ──
with sqlite3.connect(DB_FILE) as conn:
    total_q   = conn.execute('SELECT COUNT(*) FROM quotes').fetchone()[0]
    total_a   = conn.execute('SELECT COUNT(*) FROM authors').fetchone()[0]
    no_bio    = conn.execute("SELECT COUNT(*) FROM authors WHERE bio='N/A' OR bio=''").fetchone()[0]
    no_born   = conn.execute("SELECT COUNT(*) FROM authors WHERE born_date='N/A'").fetchone()[0]
    dup_q     = total_q - conn.execute('SELECT COUNT(DISTINCT quote) FROM quotes').fetchone()[0]
    dup_a     = total_a - conn.execute('SELECT COUNT(DISTINCT name)  FROM authors').fetchone()[0]

print()
print('=' * 50)
print('🔍 DATA QUALITY REPORT')
print('=' * 50)

rows = [
    ('Quotes in DB',        total_q,   None, ''),
    ('Authors in DB',       total_a,   None, ''),
    ('Duplicate quotes',    dup_q,     0,    '(should be 0)'),
    ('Duplicate authors',   dup_a,     0,    '(should be 0)'),
    ('Authors missing bio', no_bio,    0,    '(may be normal)'),
    ('Authors no born date',no_born,   0,    '(may be normal)'),
]

for label, value, expected, note in rows:
    if expected is None:
        status = 'ℹ️ '
    elif value == expected:
        status = '✅'
    else:
        status = '⚠️ '
    print(f'  {status} {label:<28}: {value:>4}  {note}')

print()
size = os.path.getsize(CSV_FILE)
print(f'CSV file size: {size:,} bytes ({size//1024} KB)')


## 📊 Final Analysis — Insights Across Both Tables

The power of a relational database: query across both tables together.

In [ ]:
# FINAL ANALYSIS

with sqlite3.connect(DB_FILE) as conn:
    conn.row_factory = sqlite3.Row

    print('=' * 55)
    print('📊 FINAL ANALYSIS')
    print('=' * 55)

    # Most quoted authors
    print('\n📌 Most quoted authors:')
    for r in conn.execute(
        'SELECT author, COUNT(*) c FROM quotes GROUP BY author ORDER BY c DESC LIMIT 5'
    ).fetchall():
        bar = '█' * r['c']
        print(f'  {r["author"]:25s} {r["c"]:2d} quotes  {bar}')

    # Most used tags
    print('\n🏷️  Collecting all tags...')
    all_tags = []
    for r in conn.execute('SELECT tags FROM quotes').fetchall():
        try:
            all_tags.extend(json.loads(r['tags']))
        except Exception:
            pass

    from collections import Counter
    top_tags = Counter(all_tags).most_common(8)
    print('   Top 8 tags:')
    for tag, count in top_tags:
        bar = '█' * count
        print(f'  {tag:20s} {count:3d}  {bar}')

    # Quotes per page
    print('\n📄 Quotes per page:')
    for r in conn.execute(
        'SELECT page_num, COUNT(*) c FROM quotes GROUP BY page_num ORDER BY page_num'
    ).fetchall():
        print(f'  Page {r["page_num"]:2d}: {r["c"]:2d} quotes')

    # Authors born by country (enrichment value!)
    print('\n🌍 Author birth locations (from enriched data):')
    for r in conn.execute(
        'SELECT born_loc, COUNT(*) c FROM authors WHERE born_loc != "N/A"'
        ' GROUP BY born_loc ORDER BY c DESC LIMIT 6'
    ).fetchall():
        print(f'  {r["born_loc"]:30s} {r["c"]} author(s)')


---
# 🎓 Course Complete — You Did It!

## What You Built in This Final Project

```
[x] Phase 1  Scraped all 100 quotes across 10 pages (paginated)
[x] Phase 2  Visited each author's detail page → bio enrichment
[x] Phase 3  Demonstrated smart_fetch Selenium fallback on JS page
[x] Phase 4  SQL JOIN across two tables → enriched dataset
[x]           Exported to CSV
[x]           Data quality report
[x]           Final analysis with cross-table insights
```

## All Skills from the Full Course

```
NOTEBOOK 1 — requests + BeautifulSoup4        (Foundation)
  requests.get(), BeautifulSoup, find/find_all,
  CSS selectors, headers, pagination, CSV/JSON save

NOTEBOOK 2 — Selenium                         (Dynamic Sites)
  WebDriver, WebDriverWait, By, click/send_keys,
  Select, iframes, alerts, screenshots, anti-detection

NOTEBOOK 3 — Production Pipelines             (Engineering)
  Session(), retry+backoff, logging, SQLite,
  deduplication, validation, checkpoint/resume,
  robots.txt, smart_fetch, context managers

NOTEBOOK 4 — E-Commerce Scraper Project       (Real Project)
  Full 1000-item scrape, tqdm progress bar,
  data quality report, multi-format export

NOTEBOOK 5 — Final Project                    (Everything Together)
  Multi-table DB, detail page enrichment,
  Selenium fallback, SQL JOIN, enriched CSV
```

## 🚀 You Are Now Ready To

```
✅ Build scrapers for any static website        (requests + BS4)
✅ Handle JavaScript-heavy sites               (Selenium)
✅ Write production-grade, crash-safe scrapers  (pipelines)
✅ Store data properly in a real database       (SQLite)
✅ Enrich data from detail pages               (multi-step)
✅ Answer interview questions on web scraping  (MAANG level)
```

## Interview Questions You Can Now Answer

| Question | Your Answer |
|----------|-------------|
| requests vs Selenium? | Static HTML → requests. JS-rendered → Selenium. Smart fetch = try both. |
| How to avoid getting blocked? | Random delays, real User-Agent, Session, anti-detection flags |
| How to make a scraper production-ready? | Retry+backoff, logging, checkpoint, validation, SQLite |
| How do you store scraped data? | SQLite with INSERT OR IGNORE for dedup; CSV for export |
| What is exponential backoff? | Wait 1s, 2s, 4s between retries — standard for flaky networks |
| What is robots.txt? | Site's rulebook for crawlers — always check before scraping |

---

> **You started from zero. You can now build production-grade web scrapers.**
> **Keep building. Keep scraping. 🚀**